# Stem cell model for solving network-related problems
Example problem: MTU (maximum transmission unit) on OSI layer 3 was set to 1400 bytes. That prohibits OSPF (a routing protocol) from reaching Full state. Because of that, layer 7 services (RabbitMQ and gRPC) are unable to exchange data. The model receives logs from:
- Cisco IOS (router setup)
- OSPF
- RabbitMQ
- gRPC

and has to transform itself from a generic model to a networking specialist, able to find causal connection between settings set on layer 3 and problems detected at layer 7.

In [1]:
from src import stem_model

settings = stem_model.Settings()
basic_agent = stem_model.StemModel("gpt-4o", settings.openai_api_key)

## 1. Collect resources to a single string

In [2]:
context_string = basic_agent.generate_context_string("resources")
print(context_string)

Source cisco-ios-console:
Core-Router-A# show running-config interface GigabitEthernet0/1
Building configuration...

Current configuration : 214 bytes
!
interface GigabitEthernet0/1
 description CONNECTION_TO_DATACENTER_CORE
 mac-address 00ab.cd12.3456
 ip address 10.254.1.1 255.255.255.252
 ip mtu 1400
 ip ospf message-digest-key 1 md5 7 0822455D0A16
 ip ospf dead-interval 40
 ip ospf hello-interval 10
 load-interval 30
 negotiation auto
end

Source gRPC-log:
[2026-05-13T17:48:10.102Z] [ERROR] [BillingSvc] gRPC call to InvoiceProcessor.SubmitBatch failed: rpc error: code = DeadlineExceeded desc = context deadline exceeded
[2026-05-13T17:48:10.105Z] [WARN] [BillingSvc] Payload size (bytes): 658402, attempt: 1/5, initiating retry in 500ms...
[2026-05-13T17:48:10.608Z] [INFO] [BillingSvc] Retrying InvoiceProcessor.SubmitBatch (correlation_id: 8f9a2b1c-4d5e-6f7a)
[2026-05-13T17:48:15.612Z] [ERROR] [BillingSvc] gRPC call to InvoiceProcessor.SubmitBatch failed: rpc error: code = DeadlineExc

## 2. First cell differentiation (first API call)

In [3]:
model_results = basic_agent.differentiate(context_string, 5)
print(model_results)

SPECIALIZATION_NAME:
Network Performance Specialist
----------------------------------------
REASONING_SUMMARY:
The environment telemetry indicates a potential issue with the network configuration affecting the gRPC service. The gRPC logs show repeated deadline exceeded errors, suggesting network latency or packet loss. The OSPF neighbor state is stuck in EXSTART, which can indicate MTU mismatches or other configuration issues. The MTU on the interface is set to 1400, which might be causing fragmentation issues for large payloads, as seen in the gRPC logs with payloads of 658402 bytes. Additionally, the high message count in the RabbitMQ queue suggests potential network congestion or processing delays.
----------------------------------------
ROOT_CAUSE_HYPOTHESIS:
The root cause of the gRPC deadline exceeded errors is likely due to MTU mismatch or suboptimal MTU settings on the GigabitEthernet0/1 interface, causing packet fragmentation and increased latency. This is supported by the O

It seems that the model correctly identified the problem and formed a correct hypothesis.

## 3. Create a specialized agent and use it to solve the problem
Here, by solving the problem, I mean selecting appropriate tools and commands for the user to paste (e.g. tool "modify router configuration" and commands for Cisco IOS).

In [4]:
specialized_agent = basic_agent.evolve()

final_results = specialized_agent.generate_remediation(context_string, 5, model_results.success_definition, model_results.root_cause_hypothesis)

print(final_results)

What to do:
1. MODIFY_ROUTER_CONFIGURATION -> 
interface GigabitEthernet0/1
 ip mtu 1500
 end

Rationale:
The current MTU setting on the GigabitEthernet0/1 interface is 1400, which is lower than the standard Ethernet MTU of 1500 bytes. This can cause packet fragmentation, leading to increased latency and potential issues with large payloads, as seen in the gRPC logs. Additionally, the OSPF neighbor is stuck in the EXSTART state, which is often caused by MTU mismatches between OSPF neighbors. By increasing the MTU to 1500, we align with the standard Ethernet MTU, reducing the likelihood of fragmentation and improving the chances of successful gRPC calls and OSPF neighbor state transitions to FULL.



Success! The advanced model correctly identified the issue and provided us a (rationalized!) solution.

## Before/after evaluation: comparison to the standard GPT-4o.
Here, I will try to just send my context string to standard GPT-4o model and see what happens.

In [9]:
from openai import OpenAI
from pydantic import BaseModel

class EvaluationAnswer(BaseModel):
    response: str

client = OpenAI(api_key=basic_agent.api_key)

prompt = f"Please solve the following problem: {context_string}"

response = client.beta.chat.completions.parse(
                    model="gpt-4o",
                    messages=[{"role": "user", "content": prompt}],
                    response_format=EvaluationAnswer
                )
print(response.choices[0].message.parsed.response)

Given the data sources provided, let's evaluate the configurations and logs to identify potential areas for resolution:

---

### Examination of Cisco IOS Console:
1. **Interface Settings**:
   - Interface **GigabitEthernet0/1** is configured correctly with an IP of **10.254.1.1/30**, and is set to **MTU 1400**.
   - The OSPF settings are:
     - **OSPF message-digest-key** is configured.
     - **Hello Interval**: 10 seconds
     - **Dead Interval**: 40 seconds
     - **State**: **DR** (Designated Router).
   
2. **Potential Issues**:
   - **MTU Mismatch**: Ensure that the MTU setting is consistent on both ends of the connection to avoid packet fragmentation and transmission issues.
   - **OSPF Neighbor State (EXSTART/BDR)**: The neighbor state should be **FULL**, indicating stable OSPF adjacency. Since it is in **EXSTART**, this may indicate a problem with the OSPF process on GigabitEthernet0/1's neighbor.

---

### Analysis of gRPC Log:
1. **Errors**: Multiple occurrences of **Deadl

As we can see, the model returned many suggestions (for all the logs, even though only one of them is really problematic). Although it mentions MTU along the way, it reaches wrong conclusions (neither network reliability nor resource optimization have anything to do with our problem).

The stem model turned out to be much better at solving this task, getting us the precise command we need to use instead of a wall of text.